# Module 6 Guided Lab: String and Categorical Data Transformation

**BAN 6003: Data Management and Analytics Integration**

In this lab, we work with a small intentionally messy business dataset. The goal is to practice practical string and categorical transformations using Pandas.

This lab focuses on:

- identifying messy text and categorical values,
- standardizing casing, spacing, and labels,
- parsing simple combined text fields,
- recoding categories using clear business rules,
- writing short notes explaining recoding assumptions.

We will **not** do heavy validation, reshaping, or aggregation in this module. Those topics appear later.

## Lab Learning Outcomes

By the end of this lab, you should be able to:

1. Identify common text and categorical data issues in business datasets.
2. Use Pandas string methods such as `.str.strip()`, `.str.lower()`, `.str.upper()`, `.str.title()`, `.str.replace()`, `.str.contains()`, and `.str.split()`.
3. Recode messy categorical values into analysis-ready categories using `.replace()` and `.map()`.
4. Explain how recoding choices affect interpretation.


## Part 0: Setup

We will use Pandas and a small customer contact dataset. The dataset is intentionally messy, but small enough to inspect directly.

In [ ]:
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

In [ ]:
raw_path = "../data/messy_customer_contacts.csv"
df_raw = pd.read_csv(raw_path)

df_raw.head(10)

## Part 1: First Look at Messy Text and Categories

Before changing text values, inspect them. Do not assume you know all possible categories.

We will use `.value_counts()` to look for:

- inconsistent capitalization,
- extra spaces,
- abbreviations,
- spelling differences,
- labels that may mean the same thing.


In [ ]:
df_raw.info()

In [ ]:
df_raw[["state_raw", "segment_raw", "contact_method_raw", "channel_raw", "product_raw"]].head(10)

In [ ]:
df_raw["state_raw"].value_counts(dropna=False)

In [ ]:
df_raw["contact_method_raw"].value_counts(dropna=False)

In [ ]:
df_raw["product_raw"].value_counts(dropna=False)

### Quick reflection

Write 2–3 short observations about the messy categories you see.

Example: `TX`, `Texas`, and `tx.` probably refer to the same state, but they appear as different values.


> Your notes here:
>
> - 
> - 
> - 

## Part 2: Keep the Raw Data and Create a Working Copy

A good habit is to keep raw columns unchanged and create cleaned columns next to them. This lets you compare before and after.

In [ ]:
df = df_raw.copy()
df.head()

## Part 3: Basic String Standardization

Common string cleaning steps include:

- `.str.strip()` to remove extra spaces at the beginning or end,
- `.str.lower()` or `.str.upper()` to standardize casing,
- `.str.title()` to make names easier to read,
- `.str.replace()` to replace punctuation or repeated spacing.

Let's start with customer names.


In [ ]:
df["customer_name_clean"] = (
    df["customer_name"]
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
    .str.title()
)

df[["customer_name", "customer_name_clean"]].head(10)

Notice that we are not changing the underlying customer identity. We are making the text easier to read and more consistent.

## Part 4: Standardize State Labels

The `state_raw` column has values like `TX`, `Texas`, `tx.`, `N.Y.`, and `New York`. We need a consistent state code.

A common workflow is:

1. strip extra spaces,
2. convert to uppercase,
3. remove periods,
4. use a mapping dictionary to convert long names to standard codes.


In [ ]:
df["state_step1"] = (
    df["state_raw"]
    .str.strip()
    .str.upper()
    .str.replace(".", "", regex=False)
)

df[["state_raw", "state_step1"]].drop_duplicates().sort_values("state_raw")

In [ ]:
state_map = {
    "TEXAS": "TX",
    "TX": "TX",
    "CALIFORNIA": "CA",
    "CA": "CA",
    "NEW YORK": "NY",
    "NY": "NY",
    "FLORIDA": "FL",
    "FL": "FL",
    "NEVADA": "NV"
}

df["state_clean"] = df["state_step1"].map(state_map)

df[["state_raw", "state_step1", "state_clean"]].drop_duplicates().sort_values("state_raw")

### Why mapping matters

The mapping dictionary is a business rule. It says which raw labels we believe are equivalent.

If a value does not appear in the mapping dictionary, Pandas will return a missing value. That is useful because it shows us which categories still need a decision.


## Part 5: Recode Contact Method Labels

The `contact_method_raw` column contains labels like `Email`, `e-mail`, `E-Mail`, `SMS`, `Text Message`, `Phone Call`, and `chat`.

We will standardize these into a smaller set of analysis-ready categories:

- `Email`
- `Phone`
- `Text`
- `Chat`


In [ ]:
df["contact_method_step1"] = (
    df["contact_method_raw"]
    .str.strip()
    .str.lower()
    .str.replace("-", "", regex=False)
    .str.replace(" ", "", regex=False)
)

df[["contact_method_raw", "contact_method_step1"]].drop_duplicates().sort_values("contact_method_raw")

In [ ]:
contact_method_map = {
    "email": "Email",
    "phone": "Phone",
    "phonecall": "Phone",
    "sms": "Text",
    "text": "Text",
    "textmessage": "Text",
    "chat": "Chat"
}

df["contact_method_clean"] = df["contact_method_step1"].map(contact_method_map)

df[["contact_method_raw", "contact_method_clean"]].drop_duplicates().sort_values("contact_method_raw")

## Part 6: Recode Customer Segments into Business Categories

Sometimes we keep detailed labels. Sometimes we group them into broader categories for analysis.

Here, we will recode customer segments into three broad groups:

- `High Value`: Premium or VIP
- `Standard`: Standard or Basic
- `Student`: Student
- `Unknown`: Unknown or not classified

This is not just a technical step. It changes how the data will be interpreted.


In [ ]:
df["segment_step1"] = df["segment_raw"].str.strip().str.lower()

segment_map = {
    "premium": "High Value",
    "vip": "High Value",
    "standard": "Standard",
    "basic": "Standard",
    "student": "Student",
    "unknown": "Unknown"
}

df["segment_group"] = df["segment_step1"].map(segment_map)

df[["segment_raw", "segment_group"]].drop_duplicates().sort_values("segment_raw")

In [ ]:
df["segment_group"].value_counts(dropna=False)

### Reflection

What assumption did we make when we grouped `premium` and `vip` together as `High Value`?

Write 1–2 sentences.


> Your answer here:
>
> 

## Part 7: Use `.str.contains()` to Create Flags

Text fields often contain useful signals. For example, the `issue_text` field may mention fraud or unauthorized charges.

We can create a simple indicator variable using `.str.contains()`.


In [ ]:
df["possible_fraud_issue"] = df["issue_text"].str.contains(
    "fraud|unauthorized",
    case=False,
    na=False
)

df[["issue_text", "possible_fraud_issue"]].head(15)

This flag does not prove fraud. It only identifies records where the text contains certain words. That distinction matters.

## Part 8: Use `.str.split()` for Simple Parsing

Sometimes one column contains multiple pieces of information. We can use `.str.split()` to separate simple combined fields.

Here, we will split the cleaned customer name into first and last names. This is just a demonstration. Real names can be more complicated than this, so this won't work perfectly in every situation.


In [ ]:
df[["first_name", "last_name"]] = df["customer_name_clean"].str.split(" ", n=1, expand=True)

df[["customer_name_clean", "first_name", "last_name"]].head(10)

## Part 9: Clean Product Categories

The `product_raw` column contains product names with underscores, inconsistent capitalization, and spacing. We will standardize these labels into readable product categories.


In [ ]:
df["product_step1"] = (
    df["product_raw"]
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)

df[["product_raw", "product_step1"]].drop_duplicates().sort_values("product_raw")

In [ ]:
product_map = {
    "loan application": "Loan Application",
    "checking": "Checking Account",
    "checking account": "Checking Account",
    "credit card": "Credit Card",
    "mortgage": "Mortgage",
    "auto loan": "Auto Loan",
    "student loan": "Student Loan",
    "personal loan": "Personal Loan"
}

df["product_clean"] = df["product_step1"].map(product_map)

df[["product_raw", "product_clean"]].drop_duplicates().sort_values("product_raw")

## Part 10: Create a Clean Analysis Dataset

Now we will create a smaller dataset with the cleaned fields. We keep important identifiers and analysis-ready categories.


In [ ]:
analysis_df = df[[
    "customer_id",
    "customer_name_clean",
    "first_name",
    "last_name",
    "state_clean",
    "segment_group",
    "contact_method_clean",
    "channel_raw",
    "product_clean",
    "possible_fraud_issue",
    "issue_text"
]].copy()

analysis_df.head(10)

## Part 11: Short Recoding Notes

Good analysts explain category decisions. This does not need to be long, but it should be clear enough that someone else can understand your assumptions.

Complete the notes below.


### Recoding Notes

1. State labels:  
   - Decision made:
   - Assumption:

2. Contact method labels:  
   - Decision made:
   - Assumption:

3. Segment grouping:  
   - Decision made:
   - Assumption:

4. Product labels:  
   - Decision made:
   - Assumption:


## Part 12: Your Turn

Complete the exercises below using the same dataset.

Use only the Pandas tools introduced in this lab.


### Exercise 1: Standardize Channel Labels

Create a new column called `channel_clean`.

Suggested categories:

- `Web Organic`
- `Web Paid`
- `Mobile App`
- `Branch`
- `Call Center`
- `Partner Referral`
- `Unknown`

Hint: start by using `.str.strip()`, `.str.lower()`, and `.str.replace()`.


In [ ]:
# Your code here

### Exercise 2: Create a Digital Channel Flag

Create a Boolean column called `digital_channel`.

Set it to `True` when the cleaned channel is one of:

- `Web Organic`
- `Web Paid`
- `Mobile App`

Otherwise, set it to `False`.

Hint: use `.isin()`.


In [ ]:
# Your code here

### Exercise 3: Create State Region Categories

Create a new column called `state_region` using `state_clean`.

Suggested mapping:

- `TX`, `FL` → `South`
- `CA`, `NV` → `West`
- `NY` → `Northeast`

Use `.map()`.


In [ ]:
# Your code here

### Exercise 4: Explain One Recoding Decision

Choose one category you recoded in this lab. Explain the decision in 2–3 sentences.

Be specific. What raw labels did you combine? Why is that reasonable? What could be lost by combining them?


> Your answer here:
>
> 

## Part 13: Optional Export

If your instructor asks you to submit a cleaned output file, run the cell below.


In [ ]:
# Uncomment and run if instructed.
# analysis_df.to_csv("../data/customer_contacts_cleaned.csv", index=False)

## Final Reminder

String and categorical transformations are not just cosmetic. They affect counts, summaries, groups, dashboards, and model features.

Before you recode categories, ask:

- What labels are truly equivalent?
- What business meaning might be lost?
- Can I explain my decision to another analyst or manager?
